In [1]:
from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Gọi Tokenizer của bge-m3
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

In [4]:
# 2. BẮT BUỘC dùng hàm .from_huggingface_tokenizer
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=600,       # Bây giờ nó mới hiểu đây là 600 TOKENS (~ 500 chữ)
    chunk_overlap=100,     # 100 TOKENS gối đầu
)

pf = pd.read_csv("../data/clean/vi-wiki.csv")
#pf = pd.read_csv("ktct.csv")
records = pf.to_dict("records")

output_file = "../data/chunk/vi-wiki.jsonl"

with open(output_file, "w", encoding="utf-8") as f:

    global_chunk_counter = 0

    for doc_index, item in enumerate(records):

        raw_text = item["text"]
        #source_name = item["source"]
        source_name = "vi-wiki"
        
        category = (
            "Giao_Trinh"
            if "Giao Trinh" in source_name
            else "Wikipedia"
        )

        chunks = text_splitter.split_text(raw_text)

        for internal_index, chunk_text in enumerate(chunks):

            safe_source_name = (
                source_name.lower()
                .replace(".pdf", "")
                .replace(" ", "_")
            )

            chunk_id = (
                f"{safe_source_name}"
                f"_doc{doc_index:03d}"
                f"_chunk{internal_index:03d}"
            )

            chunk_data = {
                "chunk_id": chunk_id,
                "text": chunk_text,
                "metadata": {
                    "source": source_name,
                    "category": category,
                    "section_title": item["title"],
                    #"page_number": item["page"],
                    "chunk_index": internal_index
                }
            }

            f.write(
                json.dumps(chunk_data, ensure_ascii=False)
                + "\n"
            )

            global_chunk_counter += 1

print(
    f"✅ Đã tạo {global_chunk_counter} chunks"
)

NameError: name 'json' is not defined

In [2]:
import json
import re
import pandas as pd
from tqdm.auto import tqdm


# ============================================================
# LOAD DATA
# ============================================================

pf = pd.read_json("../data/clean/law.json").head(10)


# ============================================================
# METADATA EXTRACTION
# ============================================================

def extract_legal_metadata(content):
    """
    Lấy Điều + tiêu đề từ dòng đầu tiên của văn bản
    """
    if not isinstance(content, str):
        return "Không xác định", "Không xác định"

    first_line = content.split("\n")[0].strip()

    match = re.search(r'(Điều\s+\d+[a-zA-Z]*)\.\s*(.*)', first_line)

    if match:
        article_number = match.group(1).strip()
        section_title = match.group(2).strip()
        return article_number, section_title

    return "Không xác định", "Không xác định"


# ============================================================
# OUTPUT FILE
# ============================================================

output_file = "../data/chunk/law-test.jsonl"


# ============================================================
# PROCESS + PROGRESS BAR
# ============================================================

with open(output_file, "w", encoding="utf-8") as f:

    for index, row in enumerate(
        tqdm(
            pf.itertuples(index=False),
            total=len(pf),
            desc="Processing legal documents",
            unit="doc"
        )
    ):

        text_content = getattr(row, "content", "")
        source_name = getattr(row, "source", "unknown")

        if not text_content:
            continue

        article_number, section_title = extract_legal_metadata(text_content)

        safe_source_name = str(source_name).lower().replace(" ", "_")

        chunk_id = f"{safe_source_name}_{index:05d}"

        chunk_data = {
            "chunk_id": chunk_id,
            "text": text_content,
            "metadata": {
                "source": source_name,
                "category": "Luat_Phap",
                "section_title": section_title,
                "article_number": article_number,
                "chunk_index": index
            }
        }

        f.write(
            json.dumps(
                chunk_data,
                ensure_ascii=False
            ) + "\n"
        )


print("✅ Đã xử lý và đóng gói thành công!")

Processing legal documents:   0%|          | 0/10 [00:00<?, ?doc/s]

✅ Đã xử lý và đóng gói thành công!


In [4]:
import json
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ============================================================
# 1. LOAD TOKENIZER & SPLITTER
# ============================================================
print("🚀 Đang tải bộ đếm Token của BAAI/bge-m3...")
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

print("✂️ Khởi tạo Text Splitter (Size: 600, Overlap: 100)...")
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=600,        
    chunk_overlap=100,
    # Cấu hình ưu tiên cắt: Cắt theo đoạn văn trước (\n\n), nếu đoạn dài quá mới cắt theo câu (.), cuối cùng mới cắt chữ
    separators=["\n\n", "\n", ". ", " ", ""] 
)

# ============================================================
# 2. CẤU HÌNH ĐẦU VÀO / ĐẦU RA
# ============================================================
input_file = "../data/clean/law.json"  
output_file = "../data/chunk/law1.jsonl"

print(f"📂 Đang đọc dữ liệu từ: {input_file}")
try:
    pf = pd.read_json(input_file)
except ValueError:
    print("❌ Lỗi định dạng JSON. Hãy đảm bảo file nguồn là JSON hợp lệ.")
    exit()

# ============================================================
# 3. XỬ LÝ & BĂM NHỎ TÀI LIỆU
# ============================================================
total_chunks_created = 0


def extract_legal_metadata(content):
    """
    Lấy Điều + tiêu đề từ dòng đầu tiên của văn bản
    """
    if not isinstance(content, str):
        return "Không xác định", "Không xác định"

    first_line = content.split("\n")[0].strip()

    match = re.search(r'(Điều\s+\d+[a-zA-Z]*)\.\s*(.*)', first_line)

    if match:
        article_number = match.group(1).strip()
        section_title = match.group(2).strip()
        return article_number, section_title

    return "Không xác định", "Không xác định"


with open(output_file, "w", encoding="utf-8") as f:
    for doc_index, row in enumerate(
        tqdm(
            pf.itertuples(index=False),
            total=len(pf),
            desc="Đang băm nhỏ tài liệu",
            unit="doc"
        )
    ):
        text_content = getattr(row, "content", "")
        source_name = getattr(row, "source", "unknown")


        if not text_content:
            continue

        article_number, section_title = extract_legal_metadata(text_content)

        if article_number != "Không xác định":
            text_content = "\n".join(text_content.split("\n")[1:]).strip()

        if not text_content:
            continue

        chunks = text_splitter.split_text(text_content)
        

        safe_source_name = str(source_name).lower().replace(" ", "_")

        # Duyệt qua từng chunk vừa được cắt ra để đóng gói
        for chunk_idx, chunk_text in enumerate(chunks):
            
            chunk_id = f"{safe_source_name}_doc{doc_index}_chunk{chunk_idx}"
            
            # Chỉ giữ lại đếm số lượng ký tự (nhẹ và nhanh bằng CPU)
            char_count = len(chunk_text)

            chunk_data = {
                "chunk_id": chunk_id,
                "text": chunk_text,
                "metadata": {
                    "source": source_name,
                    "category": "Luat_Phap",
                    "doc_index": doc_index,   
                    "chunk_index": chunk_idx, 
                    "section_title": section_title,
                    "article_number": article_number
                }
            }

            # Ghi trực tiếp xuống ổ cứng (bảo vệ RAM)
            f.write(
                json.dumps(
                    chunk_data,
                    ensure_ascii=False
                ) + "\n"
            )
            total_chunks_created += 1

print(f"\n✅ Tuyệt vời! Đã băm nhỏ thành công {len(pf)} tài liệu gốc thành {total_chunks_created} chunks!")
print(f"📁 File kết quả đã lưu tại: {output_file}")

🚀 Đang tải bộ đếm Token của BAAI/bge-m3...


✂️ Khởi tạo Text Splitter (Size: 600, Overlap: 100)...
📂 Đang đọc dữ liệu từ: ../data/clean/law.json


Đang băm nhỏ tài liệu:   0%|          | 0/19369 [00:00<?, ?doc/s]


✅ Tuyệt vời! Đã băm nhỏ thành công 19369 tài liệu gốc thành 22996 chunks!
📁 File kết quả đã lưu tại: ../data/chunk/law1.jsonl
